# Air Pollution

### Imports and preprocessing

In [6]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

ROOT_DIR = Path.cwd().parent
Tables_DIR = ROOT_DIR / "data/tables"

# helper that saves each graph + summary so src/make_pdf.py can build the PDF
sys.path.insert(0, str(ROOT_DIR / "src"))
from report import save_report

event_countriesdf = pd.read_csv(Tables_DIR / "event_countries.csv")
eventsdf          = pd.read_csv(Tables_DIR / "events.csv")
country_yearlydf  = pd.read_csv(Tables_DIR / "country_yearly.csv")
global_yearlydf   = pd.read_csv(Tables_DIR / "global_yearly.csv")
coveragedf        = pd.read_csv(Tables_DIR / "data_coverage.csv")

for name, d in [("country_yearly", country_yearlydf), ("global_yearly", global_yearlydf),
                ("events", eventsdf), ("event_countries", event_countriesdf)]:
    print(name, list(d.columns))

# ---------- column auto-detection (edit candidates if needed) ----------
def col(df, *names):
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"None of {names} found in {list(df.columns)}")

C_COUNTRY = col(country_yearlydf, "country", "country_name", "entity")
C_ISO     = col(country_yearlydf, "iso_code", "iso3", "country_code", "iso")
C_YEAR    = col(country_yearlydf, "year")
C_CO2     = col(country_yearlydf, "co2", "co2_mt", "annual_co2")
C_PC      = col(country_yearlydf, "co2_per_capita", "co2_pc", "co2_percap")
C_PM      = col(country_yearlydf, "pm25", "pm2_5", "pm25_exposure")
TOPIC = "air_pollution"
G_YEAR = col(global_yearlydf, "year")
G_TEMP = col(global_yearlydf, "temp_anomaly", "temperature_anomaly", "temp_anom", "anomaly", "j_d", "temp")

E_ID   = col(eventsdf, "event_id", "id")
E_YEAR = col(eventsdf, "year", "event_year")
E_NAME = col(eventsdf, "event_name", "name", "title", "event")
EC_ID  = col(event_countriesdf, "event_id", "id")
EC_ISO = col(event_countriesdf, "iso_code", "iso3", "country_code", "iso")

# real countries only (drop OWID aggregates like World, Asia, EU...)
is_country = (country_yearlydf[C_ISO].notna()
              & ~country_yearlydf[C_ISO].astype(str).str.startswith("OWID"))
cdf = country_yearlydf[is_country].copy()

# ---------- chart style ----------
NAVY, BLUE, RED, DARK = "#2c3e5e", "#4a78a8", "#e05252", "#2b2b2b"
FILL, GRID = "rgba(74,120,168,0.10)", "#e6eaf2"

def style(fig, title, subtitle, xt, yt, h=450, ylim=None):
    fig.update_layout(
        title=dict(text=f"<b>{title}</b><br><sup>{subtitle}</sup>",
                   x=0.02, xanchor="left", font=dict(size=20, color=NAVY)),
        template="plotly_white", width=1090, height=h,
        margin=dict(l=75, r=45, t=95, b=95),
        font=dict(color=NAVY, size=13),
        plot_bgcolor="white", paper_bgcolor="white",
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.25),
    )
    fig.update_xaxes(title=xt, gridcolor=GRID, zeroline=False)
    fig.update_yaxes(title=yt, gridcolor=GRID, zeroline=False, range=ylim)
    return fig

def dot(x, y, label, color, name, pos="top left", showlegend=True):
    return go.Scatter(x=[x], y=[y], mode="markers+text", text=[label],
                      textposition=pos, name=name, showlegend=showlegend,
                      marker=dict(size=11, color=color), textfont=dict(color=NAVY))

def world_series(value_col):
    """World total per year = sum over real countries."""
    return cdf.groupby(C_YEAR)[value_col].sum(min_count=1)

country_yearly ['iso_code', 'country', 'year', 'population', 'gdp', 'co2_mt', 'co2_per_capita', 'cumulative_co2', 'share_global_co2', 'coal_co2', 'oil_co2', 'gas_co2', 'cement_co2', 'flaring_co2', 'methane', 'nitrous_oxide', 'total_ghg', 'pm25_exposure', 'co2_yoy_pct', 'rank_total_co2', 'rank_per_capita', 'co2_per_gdp', 'gdp_change_10yr_pct', 'co2_change_10yr_pct', 'decoupled_flag']
global_yearly ['year', 'decade', 'is_complete_year', 'co2_ppm', 'co2_mlo_ppm', 'ch4_ppb', 'n2o_ppb', 'temp_anomaly', 'world_co2_mt', 'co2_mlo_yoy_change', 'co2_pct_index', 'ch4_pct_index', 'n2o_pct_index', 'temp_yoy_change', 'co2_mlo_detrended', 'temp_detrended']
events ['event_id', 'event_name', 'year', 'event_type', 'source_note']
event_countries ['event_id', 'iso_code', 'role']


### Q1. Which countries have the highest PM2.5 exposure, and how has it changed since 1990?

In [7]:
pm = cdf.dropna(subset=[C_PM]).pivot_table(index=C_COUNTRY, columns=C_YEAR, values=C_PM)
last = pm.columns.max()
first = pm.columns[pm.columns >= 1990].min()
t15 = pm[last].dropna().sort_values(ascending=False).head(15)
chg_pm = ((pm.loc[t15.index, last] / pm.loc[t15.index, first] - 1) * 100)

# ---- graph ----
fig = go.Figure(go.Bar(x=t15.values[::-1], y=t15.index[::-1], orientation="h", marker_color=BLUE,
                       text=[f"{v:.0f} µg/m³ ({chg_pm[c]:+.0f}% since {first})" for c, v in t15.items()][::-1],
                       textposition="outside", name=f"PM2.5 in {last}"))
style(fig, f"Highest PM2.5 Exposure, {last}",
      f"Top 15 countries; change since {first} shown in labels", "PM2.5 (µg/m³)", "", h=520)
fig.update_xaxes(range=[0, t15.max() * 1.45])
fig.show()

# ---- summary ----
top3 = ", ".join(f"{c} ({v:.0f} µg/m³)" for c, v in t15.head(3).items())
summary = f"""
In {last}, the highest PM2.5 exposure was in {top3},
several times the WHO guideline of 5 µg/m³. Since {first}, {int((chg_pm < 0).sum())} of these 15 countries improved
while {int((chg_pm >= 0).sum())} worsened, so high exposure is persistent and not uniformly improving.
""".strip()
print(summary)
save_report(TOPIC, "q1", "Which countries have the highest PM2.5 exposure, and how has it changed since 1990?",
            fig, summary)

In 2023, the highest PM2.5 exposure was in Qatar (108 µg/m³), Saudi Arabia (74 µg/m³), Bangladesh (69 µg/m³),
several times the WHO guideline of 5 µg/m³. Since 1990, 4 of these 15 countries improved
while 11 worsened, so high exposure is persistent and not uniformly improving.
[report] saved air_pollution/q1 (1 graph(s))


### Q2. Do countries with higher CO2 per capita have higher PM2.5?

In [8]:
yr = cdf.dropna(subset=[C_PM, C_PC]).groupby(C_YEAR).size().idxmax()
s = cdf[cdf[C_YEAR] == yr].dropna(subset=[C_PM, C_PC])
s = s[s[C_PC] > 0]
rho = s[C_PC].corr(s[C_PM], method="spearman")
m, b = np.polyfit(np.log10(s[C_PC]), s[C_PM], 1)
xs = np.linspace(s[C_PC].min(), s[C_PC].max(), 100)

# ---- graph ----
fig = go.Figure()
fig.add_trace(go.Scatter(x=s[C_PC], y=s[C_PM], mode="markers", name="Country",
                         text=s[C_COUNTRY], marker=dict(size=9, color=BLUE, opacity=0.7)))
fig.add_trace(go.Scatter(x=xs, y=m * np.log10(xs) + b, mode="lines", name=f"Trend (Spearman rho = {rho:.2f})",
                         line=dict(color=RED, width=3)))
style(fig, f"CO2 per Capita vs PM2.5 Exposure, {yr}",
      f"Spearman rho = {rho:.2f} across {len(s)} countries", "CO2 per capita (t, log scale)", "PM2.5 (µg/m³)")
fig.update_xaxes(type="log")
fig.show()

# ---- summary ----
rel = "a weak or negative" if rho < 0.2 else "a moderate positive"
explain = ("High-income, high-CO2 countries often have relatively clean air because of emission controls, while many "
           "low-emitting countries face high PM2.5 from solid-fuel cooking, dust and industry."
           if rho < 0.3 else "Higher-emitting countries tend to also have higher PM2.5.")
summary = f"""
Across {len(s)} countries in {yr}, the rank correlation is {rho:.2f}, i.e. {rel} relationship.
{explain}
CO2 per capita is therefore a poor predictor of local air quality.
""".strip()
print(summary)
save_report(TOPIC, "q2", "Do countries with higher CO2 per capita have higher PM2.5?", fig, summary)

Across 192 countries in 1994, the rank correlation is -0.22, i.e. a weak or negative relationship.
High-income, high-CO2 countries often have relatively clean air because of emission controls, while many low-emitting countries face high PM2.5 from solid-fuel cooking, dust and industry.
CO2 per capita is therefore a poor predictor of local air quality.
[report] saved air_pollution/q2 (1 graph(s))


### Q3. Where PM2.5 fell sharply, what changed (policy events)?

In [9]:
base = pm[first]
fall = ((pm[last] / base - 1) * 100)[base >= 15].dropna().sort_values().head(10)

# ---- graph ----
fig = go.Figure(go.Bar(x=fall.values[::-1], y=fall.index[::-1], orientation="h", marker_color=RED,
                       text=[f"{v:.0f}%" for v in fall.values[::-1]], textposition="outside",
                       name=f"Change {first} to {last}"))
style(fig, "Sharpest PM2.5 Declines",
      f"Countries with PM2.5 of 15+ µg/m³ in {first}", "Change in PM2.5 (%)", "", h=480)
fig.update_xaxes(range=[fall.min() * 1.3, 0])
fig.show()

iso_map = cdf.drop_duplicates(C_COUNTRY).set_index(C_COUNTRY)[C_ISO]
hits = (event_countriesdf[event_countriesdf[EC_ISO].isin(iso_map.reindex(fall.index).dropna())]
        .merge(eventsdf, left_on=EC_ID, right_on=E_ID))
ev_txt = "; ".join(sorted({f"{r[E_NAME]} ({r[E_YEAR]})" for _, r in hits.iterrows()})) or "no matching events in events.csv"

# ---- summary ----
top3 = ", ".join(f"{c} ({v:.0f}%)" for c, v in fall.head(3).items())
summary = f"""
The biggest declines since {first} were in {top3}.
Policy events recorded for these countries: {ev_txt}.
Matching an event to a decline shows timing overlap, not proof; check each country's PM2.5 curve around the event year.
""".strip()
print(summary)
save_report(TOPIC, "q3", "Where PM2.5 fell sharply, what changed (policy events)?", fig, summary)

The biggest declines since 1990 were in Bolivia (-70%), Estonia (-69%), Lithuania (-66%).
Policy events recorded for these countries: EU Emissions Trading System launched (2005); Global financial crisis (2008-09) (2008); Kyoto Protocol enters into force (2005).
Matching an event to a decline shows timing overlap, not proof; check each country's PM2.5 curve around the event year.
[report] saved air_pollution/q3 (1 graph(s))
